# AI데이터사이언스 14강 — 예측 (Prediction) ①  개요 · 상관

**2026-1학기 · ICT융합학부 컴퓨터공학트랙 · 김대환**

이 노트북은 강의자료 `AI데이터사이언스14_1.pdf` 의 흐름을 그대로 따라가는 **실습 파일**입니다.
원본 강의는 UC Berkeley *Data 8* 의 `datascience` 라이브러리(`Table`)를 사용하지만,
이 실습은 어디서나 바로 실행되도록 **numpy / pandas / matplotlib** 표준 라이브러리로 재구성했습니다.

> 이번 강의(14_1)는 목차 7개 중 **1. 개요** 와 **2. 상관(Correlation)** 까지 다룹니다.
> 회귀직선·최소제곱·진단은 다음 강의(14_2)에서 이어집니다.

## 목차
1. **개요** — 데이터로 미래 예측 (부모 키 → 자녀 키)
2. **상관 (Correlation)** — 선형 연관성의 방향과 강도
   - 표준 단위, 상관계수 r, r의 성질
   - 비선형·이상치·생태학적 상관·허위 상관

<div align="center">
<img src="images/14_1/page_01.png" width="760" alt="강의 슬라이드 p.1"><br>
<sub>📑 강의 슬라이드 p.1</sub>
</div>

<div align="center">
<img src="images/14_1/page_02.png" width="760" alt="강의 슬라이드 p.2 (목차)"><br>
<sub>📑 강의 슬라이드 p.2 (목차)</sub>
</div>

## 0. 준비 — 라이브러리 & 헬퍼 함수

아래 셀을 가장 먼저 실행하세요.

데이터셋(`family_heights.csv`, `hybrid.csv`, `sat2014.csv`)은 **이 노트북과 같은 `14장` 폴더에 이미 들어 있습니다.**
`load_data()` 는 ① 같은 폴더 → ② 현재 작업 폴더 → ③ (없을 때만) 원격 다운로드 순으로 찾으므로 **인터넷 없이도 실행**됩니다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.grid'] = True
np.random.seed(14)  # 재현성 (14장)

try:
    DATA_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    DATA_DIR = os.getcwd()  # Jupyter/Colab: 현재 작업 폴더

# 데이터셋마다 원격 출처가 달라 파일별 URL 을 둔다 (로컬에 있으면 사용 안 함)
DATA_URLS = {
    'family_heights.csv': 'https://raw.githubusercontent.com/mdogy/dataForEng1999/master/family_heights.csv',
    'hybrid.csv':         'https://raw.githubusercontent.com/data-8/materials-fa17/master/lec/hybrid.csv',
    'sat2014.csv':        'https://raw.githubusercontent.com/data-8/materials-sp18/master/lec/sat2014.csv',
}

def load_data(filename):
    """데이터셋 로드 (① 같은 폴더 → ② 작업 폴더 → ③ 원격 다운로드)."""
    local_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(local_path):
        return pd.read_csv(local_path)
    if os.path.exists(filename):
        return pd.read_csv(filename)
    url = DATA_URLS.get(filename)
    if url is None:
        raise RuntimeError(f'{filename} 의 원격 주소를 모릅니다.')
    try:
        return pd.read_csv(url)
    except Exception as e:
        raise RuntimeError(f'{filename} 를 불러올 수 없습니다: {e}')

def standard_units(arr):
    """표준 단위(z = (값 - 평균) / SD)로 변환. (Data 8 과 동일하게 모표준편차 사용)"""
    arr = np.asarray(arr, dtype=float)
    return (arr - np.mean(arr)) / np.std(arr)

def correlation(df, x, y):
    """두 열을 표준단위로 바꾼 뒤 곱의 평균 = 상관계수 r."""
    return np.mean(standard_units(df[x]) * standard_units(df[y]))

print('준비 완료 ✔  (데이터 폴더:', DATA_DIR, ')')

## 1. 개요 — 데이터로 미래 예측

한 변수의 값으로 **다른 변수의 값을 예측**하는 가장 일반적인 방법을 배웁니다.

**예제: 부모 키로 자녀 키 예측 (934명)**
부모의 평균 키(`MidParent`, 중간부모키)와 다 자란 자녀의 키(`Child`) 데이터입니다.
원본 컬럼명 `midparentHeight`, `childHeight` 를 보기 쉽게 `MidParent`, `Child` 로 바꿉니다.

<div align="center">
<img src="images/14_1/page_03.png" width="760" alt="강의 슬라이드 p.3 (개요)"><br>
<sub>📑 강의 슬라이드 p.3 (개요)</sub>
</div>

<div align="center">
<img src="images/14_1/page_04.png" width="760" alt="강의 슬라이드 p.4"><br>
<sub>📑 강의 슬라이드 p.4</sub>
</div>

In [ ]:
# 부모-자녀 키 데이터 (Galton)
original = load_data('family_heights.csv')
heights = pd.DataFrame({
    'MidParent': original['midparentHeight'],
    'Child':     original['childHeight'],
})
print('자녀 수:', len(heights))
heights.head()

In [ ]:
# 산점도: 중간부모키 vs 자녀키
heights.plot.scatter('MidParent', 'Child', s=8, alpha=0.5)
plt.title('부모 키 vs 자녀 키 (934명)')
plt.show()

**예측 방법:** 중간부모키가 `mp ± 0.5`인치인 자녀들만 모아 그 **평균 키**를 예측값으로 삼습니다.
즉, 각 x 위치에서 **수직 띠(strip)의 중심**을 잇는 것이 예측입니다.

➡️ 이렇게 수직 띠의 중심을 잇는 예측 방법을 **회귀(Regression)** 라고 합니다.

<div align="center">
<img src="images/14_1/page_05.png" width="760" alt="강의 슬라이드 p.5"><br>
<sub>📑 강의 슬라이드 p.5</sub>
</div>

In [ ]:
def predict_child(mp):
    """중간부모키가 mp ± 0.5 인치인 자녀들의 평균 키를 반환."""
    nearby = heights[(heights['MidParent'] >= mp - 0.5) &
                     (heights['MidParent'] <= mp + 0.5)]
    return nearby['Child'].mean()

# 모든 중간부모키에 예측 적용
heights['Prediction'] = heights['MidParent'].apply(predict_child)
heights.head()

In [ ]:
# 원본 산점도 위에 예측값(주황)을 겹쳐 그리기
ax = heights.plot.scatter('MidParent', 'Child', s=8, alpha=0.4, label='Child')
heights.plot.scatter('MidParent', 'Prediction', s=8, color='gold',
                     ax=ax, label='Prediction')
plt.legend()
plt.title('수직 띠의 중심을 잇는 예측 = 회귀')
plt.show()

## 2. 상관 (Correlation)

**선형 연관성(linear association)** — 산점도가 직선 주위에 얼마나 조밀하게 모여 있는지를 측정합니다.

<div align="center">
<img src="images/14_1/page_06.png" width="760" alt="강의 슬라이드 p.6 (상관)"><br>
<sub>📑 강의 슬라이드 p.6 (상관)</sub>
</div>

### 2-1. 예제 데이터: 하이브리드 차량 (1997~2013)

| 열 | 의미 |
|---|---|
| `vehicle` | 차량 모델 |
| `year` | 제조 연도 |
| `msrp` | 권장 소비자가격 (2013 달러) |
| `acceleration` | 가속도 (km/h per second) |
| `mpg` | 연비 (miles per gallon) |
| `class` | 차종 |

<div align="center">
<img src="images/14_1/page_07.png" width="760" alt="강의 슬라이드 p.7 (상관 1)"><br>
<sub>📑 강의 슬라이드 p.7 (상관 1)</sub>
</div>

In [ ]:
hybrid = load_data('hybrid.csv')
print(hybrid.shape)
hybrid.head(10)

### 2-2. 산점도로 연관성 보기

방향/모양만으로도 연관성을 이해할 수 있습니다.
- **가속도 vs 가격**: 양(positive)의 연관 (우상향)
- **연비 vs 가격**: 음(negative)의 연관 + 비선형(곡선)
- **SUV로 한정한 연비 vs 가격**: 음의 연관

<div align="center">
<img src="images/14_1/page_08.png" width="760" alt="강의 슬라이드 p.8 (상관 2)"><br>
<sub>📑 강의 슬라이드 p.8 (상관 2)</sub>
</div>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

hybrid.plot.scatter('acceleration', 'msrp', s=10, ax=axes[0])
axes[0].set_title('가속도 vs 가격 (양의 연관)')

hybrid.plot.scatter('mpg', 'msrp', s=10, ax=axes[1])
axes[1].set_title('연비 vs 가격 (음 + 비선형)')

suv = hybrid[hybrid['class'] == 'SUV']
suv.plot.scatter('mpg', 'msrp', s=10, color='darkred', ax=axes[2])
axes[2].set_title('SUV: 연비 vs 가격 (음의 연관)')

plt.tight_layout()
plt.show()

### 2-3. 표준 단위로 변환

측정 단위에 신경 쓰지 않아도 되고, 동일한 축 척도에서 비교할 수 있습니다.
표준 단위 `z = (값 - 평균) / SD` 로 바꾸면 산점도 모양은 같지만 축이 -3~3 범위가 됩니다.

<div align="center">
<img src="images/14_1/page_09.png" width="760" alt="강의 슬라이드 p.9 (상관 3)"><br>
<sub>📑 강의 슬라이드 p.9 (상관 3)</sub>
</div>

In [ ]:
# SUV 의 연비/가격을 표준 단위로 변환해 산점도
su = pd.DataFrame({
    'mpg (su)':  standard_units(suv['mpg']),
    'msrp (su)': standard_units(suv['msrp']),
})
su.plot.scatter('mpg (su)', 'msrp (su)', s=15)
plt.xlim(-3, 3); plt.ylim(-3, 3)
plt.title('SUV: 표준 단위로 본 연비 vs 가격')
plt.show()

### 2-4. 상관 계수 (Correlation coefficient) r

- 두 변수 사이의 **선형** 관계의 **강도**를 측정
- 산점도가 직선 주위에 얼마나 모여 있는지를 나타냄
- 기호 **r**, 항상 **-1 ≤ r ≤ 1**
  - `r = 1` : 완벽한 양의 직선 (우상향)
  - `r = -1` : 완벽한 음의 직선 (우하향)
  - `r = 0` : 선형 관계 없음

아래에서 r 값을 바꿔가며 산점도 모양이 어떻게 달라지는지 확인합니다.

<div align="center">
<img src="images/14_1/page_10.png" width="760" alt="강의 슬라이드 p.10 (상관 4)"><br>
<sub>📑 강의 슬라이드 p.10 (상관 4)</sub>
</div>

In [ ]:
def r_scatter(r, n=500):
    """주어진 상관계수 r 에 가까운 표본 산점도를 생성."""
    x = np.random.normal(0, 1, n)
    z = np.random.normal(0, 1, n)
    y = r * x + np.sqrt(1 - r**2) * z
    return x, y

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, r in zip(axes, [0.9, 0.5, 0.0, -0.7]):
    x, y = r_scatter(r)
    ax.scatter(x, y, s=8, alpha=0.5)
    ax.set_title(f'r = {r}')
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
plt.tight_layout()
plt.show()

### 2-5. 상관 계수 직접 계산하기

**r 의 정의:** 두 변수를 모두 표준단위로 바꾼 뒤, **두 표준단위 곱의 평균**.

작은 예제로 3단계를 따라가 봅니다.
1. 각 변수를 표준 단위로 변환
2. 표준 단위 쌍을 서로 곱함
3. 곱들의 평균을 구함

<div align="center">
<img src="images/14_1/page_11.png" width="760" alt="강의 슬라이드 p.11 (상관 5)"><br>
<sub>📑 강의 슬라이드 p.11 (상관 5)</sub>
</div>

In [ ]:
t = pd.DataFrame({'x': np.arange(1, 7, 1),
                  'y': np.array([2, 3, 1, 5, 2, 7])})
t.plot.scatter('x', 'y', s=40, color='red')
plt.title('작은 예제 데이터')
plt.show()
t

In [ ]:
# 1~3 단계를 한 표에서 확인
t_calc = t.copy()
t_calc['x (su)'] = standard_units(t['x'])
t_calc['y (su)'] = standard_units(t['y'])
t_calc['product'] = t_calc['x (su)'] * t_calc['y (su)']
display(t_calc)

r = t_calc['product'].mean()   # 3단계: 곱들의 평균
print('r =', r)

### 2-6. 상관 계수의 성질

- r 은 **단위가 없는 순수한 수** (표준 단위 기반)
- 어느 축의 **단위를 바꿔도** r 은 영향받지 않음
- 두 **축을 서로 바꿔도** r 은 변하지 않음
  - 표준단위의 곱은 어느 쪽을 x/y 라 하든 동일하기 때문

<div align="center">
<img src="images/14_1/page_12.png" width="760" alt="강의 슬라이드 p.12 (상관 6)"><br>
<sub>📑 강의 슬라이드 p.12 (상관 6)</sub>
</div>

In [ ]:
# 축을 바꿔도 r 은 동일
print('correlation(t, x, y) =', correlation(t, 'x', 'y'))
print('correlation(t, y, x) =', correlation(t, 'y', 'x'))

### 2-7. `correlation` 함수로 하이브리드 데이터 분석

- 가격–연비는 **음의 연관**, 가격–가속은 **양의 연관**
- 가격–가속의 선형 관계(r ≈ 0.5)가 가격–연비(r ≈ -0.67)보다 조금 더 약함

**⚠️ 연관은 인과가 아님(Correlation ≠ Causation).** 상관은 오직 연관만 측정합니다.
(예: 아이들의 체중과 수학 실력에 양의 상관이 있지만, 살이 쪄서 수학을 잘하는 것이 아니라
**나이**가 교란 변수입니다.)

<div align="center">
<img src="images/14_1/page_13.png" width="760" alt="강의 슬라이드 p.13 (상관 7)"><br>
<sub>📑 강의 슬라이드 p.13 (상관 7)</sub>
</div>

In [ ]:
print('가격 vs 연비   r =', correlation(suv, 'mpg', 'msrp'))
print('가격 vs 가속도 r =', correlation(suv, 'acceleration', 'msrp'))

### 2-8. 상관은 **선형** 연관만 측정

강한 비선형 관계를 가진 변수들도 상관은 매우 낮을 수 있습니다.
아래 `y = x²` 는 완벽한 이차 관계지만 상관은 거의 0 입니다.

<div align="center">
<img src="images/14_1/page_14.png" width="760" alt="강의 슬라이드 p.14 (상관 8)"><br>
<sub>📑 강의 슬라이드 p.14 (상관 8)</sub>
</div>

In [ ]:
new_x = np.arange(-4, 4.1, 0.5)
nonlinear = pd.DataFrame({'x': new_x, 'y': new_x**2})
nonlinear.plot.scatter('x', 'y', s=40, color='r')
plt.title('완벽한 이차 관계 y = x²')
plt.show()

print('correlation =', correlation(nonlinear, 'x', 'y'))  # ≈ 0

### 2-9. 상관은 **이상치(Outlier)** 에 영향 받음

이상치 하나가 상관에 큰 영향을 줄 수 있습니다.
아래는 `r = 1` 이던 직선 산점도에 **단 하나의 이상점**을 추가하자 `r = 0` 이 되는 예입니다.

<div align="center">
<img src="images/14_1/page_15.png" width="760" alt="강의 슬라이드 p.15 (상관 9)"><br>
<sub>📑 강의 슬라이드 p.15 (상관 9)</sub>
</div>

In [ ]:
line = pd.DataFrame({'x': [1, 2, 3, 4], 'y': [1, 2, 3, 4]})
outlier = pd.DataFrame({'x': [1, 2, 3, 4, 5], 'y': [1, 2, 3, 4, 0]})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
line.plot.scatter('x', 'y', s=40, color='r', ax=axes[0])
axes[0].set_title(f"직선만:  r = {correlation(line, 'x', 'y'):.1f}")
outlier.plot.scatter('x', 'y', s=40, color='r', ax=axes[1])
axes[1].set_title(f"이상치 추가:  r = {correlation(outlier, 'x', 'y'):.1f}")
plt.tight_layout()
plt.show()

### 2-10. 생태학적 상관은 신중히 해석

**집계(aggregate)** 된 데이터에 기반한 상관은 오해를 부를 수 있습니다.
예: 2014년 미국 주(state)별 SAT 비판적 읽기 vs 수학 점수는 강한 상관을 보이지만,
이는 **주별 평균** 데이터이기 때문입니다. 실제 개별 학생 데이터를 보면 상관이 더 낮아집니다.

<div align="center">
<img src="images/14_1/page_16.png" width="760" alt="강의 슬라이드 p.16 (상관 10)"><br>
<sub>📑 강의 슬라이드 p.16 (상관 10)</sub>
</div>

In [ ]:
sat = load_data('sat2014.csv')
display(sat.head())

sat.plot.scatter('Critical Reading', 'Math', s=20)
plt.title("2014 SAT (주별 평균)")
plt.show()
print('주별 평균 상관 r =', correlation(sat, 'Critical Reading', 'Math'))

### 2-11. 진지한 연구? 농담?

> 국가별 1인당 **초콜릿 소비량**과 인구당 **노벨상 수상자 수** 사이에 r ≈ 0.79 의 강한 상관 (P < 0.0001)

이는 **허위 상관(spurious correlation)** 의 대표적 예입니다.
초콜릿을 먹는다고 노벨상을 받는 것이 아니라, 두 변수 모두 **국가의 부(富)** 같은
숨은 변수와 연관되어 있을 뿐입니다. **상관은 인과가 아님**을 다시 기억하세요.

<div align="center">
<img src="images/14_1/page_17.png" width="760" alt="강의 슬라이드 p.17 (상관 11)"><br>
<sub>📑 강의 슬라이드 p.17 (상관 11)</sub>
</div>

## 정리

| 개념 | 핵심 |
|---|---|
| **예측/회귀** | x 의 수직 띠 안 y 값들의 평균으로 예측 |
| **상관계수 r** | 두 변수를 표준단위로 바꾼 뒤 곱의 평균, -1 ≤ r ≤ 1 |
| **r 의 성질** | 단위 없음, 축 단위·순서 바꿔도 불변 |
| **주의 1** | r 은 **선형** 관계만 측정 (비선형은 0 일 수 있음) |
| **주의 2** | **이상치** 에 민감 |
| **주의 3** | **상관 ≠ 인과** (교란변수·생태학적·허위 상관) |

다음 강의(**14_2**)에서는 이 상관계수를 이용해 **회귀 직선(Regression Line)** 의 기울기·절편을 구하고,
**최소제곱법(Least Squares)** 으로 예측 오차를 최소화하는 직선을 찾는 방법을 배웁니다.